In [3]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/praneetharamagiri/thyroid/allhypo.data
/kaggle/input/datasets/praneetharamagiri/thyroid/allhyper.data


In [4]:
import pandas as pd

hypo_path = "/kaggle/input/datasets/praneetharamagiri/thyroid/allhypo.data"
hyper_path = "/kaggle/input/datasets/praneetharamagiri/thyroid/allhyper.data"

hypo_df = pd.read_csv(
    hypo_path,
    sep=",",
    header=None
)

hyper_df = pd.read_csv(
    hyper_path,
    sep=",",
    header=None
)

print("Hypothyroid dataset shape:", hypo_df.shape)
print("Hyperthyroid dataset shape:", hyper_df.shape)

print("\nHypothyroid first 5 rows:")
display(hypo_df.head())

print("\nHyperthyroid first 5 rows:")
display(hyper_df.head())

print("\nHypothyroid labels:")
print(hypo_df.iloc[:, -1].value_counts())

print("\nHyperthyroid labels:")
print(hyper_df.iloc[:, -1].value_counts())

Hypothyroid dataset shape: (2800, 30)
Hyperthyroid dataset shape: (2800, 30)

Hypothyroid first 5 rows:


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,41,F,f,f,f,f,f,f,f,f,...,t,125,t,1.14,t,109,f,?,SVHC,negative.|3733
1,23,F,f,f,f,f,f,f,f,f,...,t,102,f,?,f,?,f,?,other,negative.|1442
2,46,M,f,f,f,f,f,f,f,f,...,t,109,t,0.91,t,120,f,?,other,negative.|2965
3,70,F,t,f,f,f,f,f,f,f,...,t,175,f,?,f,?,f,?,other,negative.|806
4,70,F,f,f,f,f,f,f,f,f,...,t,61,t,0.87,t,70,f,?,SVI,negative.|2807



Hyperthyroid first 5 rows:


,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,41,F,f,f,f,f,f,f,f,f,...,t,125,t,1.14,t,109,f,?,SVHC,negative.|3733
1,23,F,f,f,f,f,f,f,f,f,...,t,102,f,?,f,?,f,?,other,negative.|1442
2,46,M,f,f,f,f,f,f,f,f,...,t,109,t,0.91,t,120,f,?,other,negative.|2965
3,70,F,t,f,f,f,f,f,f,f,...,t,175,f,?,f,?,f,?,other,negative.|806
4,70,F,f,f,f,f,f,f,f,f,...,t,61,t,0.87,t,70,f,?,SVI,negative.|2807



Hypothyroid labels:
29
negative.|724     1
negative.|3733    1
negative.|1442    1
negative.|2965    1
negative.|806     1
                 ..
negative.|2297    1
negative.|3564    1
negative.|152     1
negative.|936     1
negative.|716     1
Name: count, Length: 2800, dtype: int64

Hyperthyroid labels:
29
negative.|724     1
negative.|3733    1
negative.|1442    1
negative.|2965    1
negative.|806     1
                 ..
negative.|2297    1
negative.|3564    1
negative.|152     1
negative.|936     1
negative.|716     1
Name: count, Length: 2800, dtype: int64


In [5]:
hypo_labels = hypo_df.iloc[:, -1].astype(str).str.split("|").str[0]
hyper_labels = hyper_df.iloc[:, -1].astype(str).str.split("|").str[0]

print("Hypothyroid classes:")
print(hypo_labels.value_counts())

print("\nHyperthyroid classes:")
print(hyper_labels.value_counts())

Hypothyroid classes:
29
negative.                   2580
compensated hypothyroid.     154
primary hypothyroid.          64
secondary hypothyroid.         2
Name: count, dtype: int64

Hyperthyroid classes:
29
negative.        2723
hyperthyroid.      62
T3 toxic.           8
goitre.             7
Name: count, dtype: int64


## Thyroid Class Analysis

The original UCI thyroid datasets contain multiple thyroid condition subtypes. For this project, clinically related hypothyroid subtypes are grouped into a single Hypothyroid class, while hyperthyroid and T3 toxic cases are grouped into a Hyperthyroid class.

Negative cases are treated as Normal. Goitre cases are excluded because goitre does not directly represent a hyperthyroid classification.

The final target classes are:

- Normal
- Hypothyroid
- Hyperthyroid

In [6]:
# Extract actual class labels by removing the case ID
hypo_df["Class"] = (
    hypo_df.iloc[:, -1]
    .astype(str)
    .str.split("|")
    .str[0]
)

hyper_df["Class"] = (
    hyper_df.iloc[:, -1]
    .astype(str)
    .str.split("|")
    .str[0]
)

print("Hypothyroid dataset classes:")
print(hypo_df["Class"].value_counts())

print("\nHyperthyroid dataset classes:")
print(hyper_df["Class"].value_counts())

Hypothyroid dataset classes:
Class
negative.                   2580
compensated hypothyroid.     154
primary hypothyroid.          64
secondary hypothyroid.         2
Name: count, dtype: int64

Hyperthyroid dataset classes:
Class
negative.        2723
hyperthyroid.      62
T3 toxic.           8
goitre.             7
Name: count, dtype: int64


In [7]:
# Select Normal cases
normal_df = hypo_df[
    hypo_df["Class"] == "negative."
].copy()

normal_df["Thyroid_Class"] = "Normal"


# Select Hypothyroid cases
hypothyroid_labels = [
    "compensated hypothyroid.",
    "primary hypothyroid.",
    "secondary hypothyroid."
]

hypothyroid_df = hypo_df[
    hypo_df["Class"].isin(hypothyroid_labels)
].copy()

hypothyroid_df["Thyroid_Class"] = "Hypothyroid"


# Select Hyperthyroid cases
hyperthyroid_labels = [
    "hyperthyroid.",
    "T3 toxic."
]

hyperthyroid_df = hyper_df[
    hyper_df["Class"].isin(hyperthyroid_labels)
].copy()

hyperthyroid_df["Thyroid_Class"] = "Hyperthyroid"


# Combine the three classes
thyroid_df = pd.concat(
    [
        normal_df,
        hypothyroid_df,
        hyperthyroid_df
    ],
    ignore_index=True
)


print(thyroid_df["Thyroid_Class"].value_counts())

Thyroid_Class
Normal          2580
Hypothyroid      220
Hyperthyroid      70
Name: count, dtype: int64


## Data Preprocessing


In [8]:
# Check dataset information

print(thyroid_df.shape)

thyroid_df.info()

(2870, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2870 entries, 0 to 2869
Data columns (total 32 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   0              2870 non-null   object
 1   1              2870 non-null   object
 2   2              2870 non-null   object
 3   3              2870 non-null   object
 4   4              2870 non-null   object
 5   5              2870 non-null   object
 6   6              2870 non-null   object
 7   7              2870 non-null   object
 8   8              2870 non-null   object
 9   9              2870 non-null   object
 10  10             2870 non-null   object
 11  11             2870 non-null   object
 12  12             2870 non-null   object
 13  13             2870 non-null   object
 14  14             2870 non-null   object
 15  15             2870 non-null   object
 16  16             2870 non-null   object
 17  17             2870 non-null   object
 18  18             28

In [9]:
feature_names = [
    "Age",
    "Sex",
    "On_Thyroxine",
    "Query_On_Thyroxine",
    "On_Antithyroid_Medication",
    "Sick",
    "Pregnant",
    "Thyroid_Surgery",
    "I131_Treatment",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Lithium",
    "Goitre",
    "Tumor",
    "Hypopituitary",
    "Psych",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI",
    "TBG_Measured",
    "TBG",
    "Referral_Source",
    "Feature_24",
    "Feature_25",
    "Feature_26",
    "Feature_27",
    "Feature_28",
    "Original_Label"
]

thyroid_df.columns = feature_names + ["Class", "Thyroid_Class"]

thyroid_df.head()

,Age,Sex,On_Thyroxine,Query_On_Thyroxine,On_Antithyroid_Medication,Sick,Pregnant,Thyroid_Surgery,I131_Treatment,Query_Hypothyroid,...,TBG,Referral_Source,Feature_24,Feature_25,Feature_26,Feature_27,Feature_28,Original_Label,Class,Thyroid_Class
0,41,F,f,f,f,f,f,f,f,f,...,t,1.14,t,109,f,?,SVHC,negative.|3733,negative.,Normal
1,23,F,f,f,f,f,f,f,f,f,...,f,?,f,?,f,?,other,negative.|1442,negative.,Normal
2,46,M,f,f,f,f,f,f,f,f,...,t,0.91,t,120,f,?,other,negative.|2965,negative.,Normal
3,70,F,t,f,f,f,f,f,f,f,...,f,?,f,?,f,?,other,negative.|806,negative.,Normal
4,70,F,f,f,f,f,f,f,f,f,...,t,0.87,t,70,f,?,SVI,negative.|2807,negative.,Normal


In [10]:
# missing values

# Count missing values represented by '?'

missing_values = (thyroid_df == "?").sum()

missing_values[missing_values > 0]


Age                   1
Sex                 115
T3                  292
T4U                 594
TBG_Measured        184
Referral_Source     298
Feature_25          296
Feature_27         2870
dtype: int64

In [11]:
selected_features = [
    "Age",
    "Sex",
    "On_Thyroxine",
    "On_Antithyroid_Medication",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Thyroid_Surgery",
    "I131_Treatment",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI",
    "Thyroid_Class"
]


thyroid_model_df = thyroid_df[selected_features].copy()

print(thyroid_model_df.shape)

thyroid_model_df.head()

(2870, 14)


,Age,Sex,On_Thyroxine,On_Antithyroid_Medication,Query_Hypothyroid,Query_Hyperthyroid,Thyroid_Surgery,I131_Treatment,TSH,T3,TT4,T4U,FTI,Thyroid_Class
0,41,F,f,f,f,f,f,f,t,1.3,t,2.5,t,Normal
1,23,F,f,f,f,f,f,f,t,4.1,t,2,t,Normal
2,46,M,f,f,f,f,f,f,t,0.98,f,?,t,Normal
3,70,F,t,f,f,f,f,f,t,0.16,t,1.9,t,Normal
4,70,F,f,f,f,f,f,f,t,0.72,t,1.2,t,Normal


In [12]:
thyroid_model_df[
    [
        "TSH",
        "T3",
        "TT4",
        "T4U",
        "FTI"
    ]
].head(10)




,TSH,T3,TT4,T4U,FTI
0,t,1.3,t,2.5,t
1,t,4.1,t,2,t
2,t,0.98,f,?,t
3,t,0.16,t,1.9,t
4,t,0.72,t,1.2,t
5,t,0.03,f,?,t
6,f,?,f,?,t
7,t,2.2,t,0.6,t
8,t,0.6,t,2.2,t
9,t,2.4,t,1.6,t


In [13]:
thyroid_df.iloc[:5, :30].T

,0,1,2,3,4
Age,41,23,46,70,70
Sex,F,F,M,F,F
On_Thyroxine,f,f,f,t,f
Query_On_Thyroxine,f,f,f,f,f
On_Antithyroid_Medication,f,f,f,f,f
Sick,f,f,f,f,f
Pregnant,f,f,f,f,f
Thyroid_Surgery,f,f,f,f,f
I131_Treatment,f,f,f,f,f
Query_Hypothyroid,f,f,f,f,f


In [14]:
correct_columns = [
    "Age",
    "Sex",
    "On_Thyroxine",
    "Query_On_Thyroxine",
    "On_Antithyroid_Medication",
    "Sick",
    "Pregnant",
    "Thyroid_Surgery",
    "I131_Treatment",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Lithium",
    "Goitre",
    "Tumor",
    "Hypopituitary",
    "Psych",
    "TSH_Measured",
    "TSH",
    "T3_Measured",
    "T3",
    "TT4_Measured",
    "TT4",
    "T4U_Measured",
    "T4U",
    "FTI_Measured",
    "FTI",
    "TBG_Measured",
    "TBG",
    "Referral_Source",
    "Feature_26",
    "Original_Label",
    "Thyroid_Class"
]

thyroid_df.columns = correct_columns

thyroid_df.head()

,Age,Sex,On_Thyroxine,Query_On_Thyroxine,On_Antithyroid_Medication,Sick,Pregnant,Thyroid_Surgery,I131_Treatment,Query_Hypothyroid,...,T4U_Measured,T4U,FTI_Measured,FTI,TBG_Measured,TBG,Referral_Source,Feature_26,Original_Label,Thyroid_Class
0,41,F,f,f,f,f,f,f,f,f,...,t,1.14,t,109,f,?,SVHC,negative.|3733,negative.,Normal
1,23,F,f,f,f,f,f,f,f,f,...,f,?,f,?,f,?,other,negative.|1442,negative.,Normal
2,46,M,f,f,f,f,f,f,f,f,...,t,0.91,t,120,f,?,other,negative.|2965,negative.,Normal
3,70,F,t,f,f,f,f,f,f,f,...,f,?,f,?,f,?,other,negative.|806,negative.,Normal
4,70,F,f,f,f,f,f,f,f,f,...,t,0.87,t,70,f,?,SVI,negative.|2807,negative.,Normal


In [15]:
thyroid_df[["TSH","T3","TT4","T4U","FTI"]].head()

,TSH,T3,TT4,T4U,FTI
0,1.3,2.5,125,1.14,109
1,4.1,2,102,?,?
2,0.98,?,109,0.91,120
3,0.16,1.9,175,?,?
4,0.72,1.2,61,0.87,70


In [16]:
selected_features = [
    "Age",
    "Sex",
    "On_Thyroxine",
    "On_Antithyroid_Medication",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Thyroid_Surgery",
    "I131_Treatment",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI",
    "Thyroid_Class"
]

thyroid_model_df = thyroid_df[selected_features].copy()

print(thyroid_model_df.shape)

thyroid_model_df.head()

(2870, 14)


,Age,Sex,On_Thyroxine,On_Antithyroid_Medication,Query_Hypothyroid,Query_Hyperthyroid,Thyroid_Surgery,I131_Treatment,TSH,T3,TT4,T4U,FTI,Thyroid_Class
0,41,F,f,f,f,f,f,f,1.3,2.5,125,1.14,109,Normal
1,23,F,f,f,f,f,f,f,4.1,2,102,?,?,Normal
2,46,M,f,f,f,f,f,f,0.98,?,109,0.91,120,Normal
3,70,F,t,f,f,f,f,f,0.16,1.9,175,?,?,Normal
4,70,F,f,f,f,f,f,f,0.72,1.2,61,0.87,70,Normal


# Handling missing values

The dataset represents missing values using the '?' symbol. These values are converted into NaN and handled separately based on feature type.

- Numerical thyroid measurements are filled using median values.
- Categorical patient information is filled using the most frequent category.

In [17]:
import numpy as np

# Replace ? with NaN
thyroid_model_df = thyroid_model_df.replace("?", np.nan)

# Check missing values
missing = thyroid_model_df.isnull().sum()

missing[missing > 0]

Age      1
Sex    115
TSH    292
T3     594
TT4    184
T4U    298
FTI    296
dtype: int64

## Imputing Missing Values

Missing numerical thyroid measurements are filled using median values to reduce the influence of outliers. Missing categorical information is filled using the most frequent category.

In [18]:
from sklearn.impute import SimpleImputer

# Separate numerical and categorical columns

numerical_features = [
    "Age",
    "TSH",
    "T3",
    "TT4",
    "T4U",
    "FTI"
]

categorical_features = [
    "Sex",
    "On_Thyroxine",
    "On_Antithyroid_Medication",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Thyroid_Surgery",
    "I131_Treatment"
]


# Numerical imputer
num_imputer = SimpleImputer(strategy="median")

thyroid_model_df[numerical_features] = num_imputer.fit_transform(
    thyroid_model_df[numerical_features]
)


# Categorical imputer
cat_imputer = SimpleImputer(strategy="most_frequent")

thyroid_model_df[categorical_features] = cat_imputer.fit_transform(
    thyroid_model_df[categorical_features]
)


# Check missing values again

thyroid_model_df.isnull().sum()

Age                          0
Sex                          0
On_Thyroxine                 0
On_Antithyroid_Medication    0
Query_Hypothyroid            0
Query_Hyperthyroid           0
Thyroid_Surgery              0
I131_Treatment               0
TSH                          0
T3                           0
TT4                          0
T4U                          0
FTI                          0
Thyroid_Class                0
dtype: int64

## Encoding Categorical Features

In [19]:
thyroid_model_df.head()

,Age,Sex,On_Thyroxine,On_Antithyroid_Medication,Query_Hypothyroid,Query_Hyperthyroid,Thyroid_Surgery,I131_Treatment,TSH,T3,TT4,T4U,FTI,Thyroid_Class
0,41.0,F,f,f,f,f,f,f,1.30,2.5,125.0,1.14,109.0,Normal
1,23.0,F,f,f,f,f,f,f,4.10,2.0,102.0,0.98,108.0,Normal
2,46.0,M,f,f,f,f,f,f,0.98,2.0,109.0,0.91,120.0,Normal
3,70.0,F,t,f,f,f,f,f,0.16,1.9,175.0,0.98,108.0,Normal
4,70.0,F,f,f,f,f,f,f,0.72,1.2,61.0,0.87,70.0,Normal


In [20]:
from sklearn.preprocessing import LabelEncoder

# Binary columns
binary_features = [
    "On_Thyroxine",
    "On_Antithyroid_Medication",
    "Query_Hypothyroid",
    "Query_Hyperthyroid",
    "Thyroid_Surgery",
    "I131_Treatment"
]

# Convert t/f to 1/0
for col in binary_features:
    thyroid_model_df[col] = thyroid_model_df[col].map({
        "t": 1,
        "f": 0
    })


# Encode Sex
thyroid_model_df["Sex"] = thyroid_model_df["Sex"].map({
    "M": 1,
    "F": 0
})


# Encode target
label_encoder = LabelEncoder()

thyroid_model_df["Thyroid_Class"] = label_encoder.fit_transform(
    thyroid_model_df["Thyroid_Class"]
)


thyroid_model_df.head()

,Age,Sex,On_Thyroxine,On_Antithyroid_Medication,Query_Hypothyroid,Query_Hyperthyroid,Thyroid_Surgery,I131_Treatment,TSH,T3,TT4,T4U,FTI,Thyroid_Class
0,41.0,0,0,0,0,0,0,0,1.30,2.5,125.0,1.14,109.0,2
1,23.0,0,0,0,0,0,0,0,4.10,2.0,102.0,0.98,108.0,2
2,46.0,1,0,0,0,0,0,0,0.98,2.0,109.0,0.91,120.0,2
3,70.0,0,1,0,0,0,0,0,0.16,1.9,175.0,0.98,108.0,2
4,70.0,0,0,0,0,0,0,0,0.72,1.2,61.0,0.87,70.0,2


In [21]:

thyroid_model_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2870 entries, 0 to 2869
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Age                        2870 non-null   float64
 1   Sex                        2870 non-null   int64  
 2   On_Thyroxine               2870 non-null   int64  
 3   On_Antithyroid_Medication  2870 non-null   int64  
 4   Query_Hypothyroid          2870 non-null   int64  
 5   Query_Hyperthyroid         2870 non-null   int64  
 6   Thyroid_Surgery            2870 non-null   int64  
 7   I131_Treatment             2870 non-null   int64  
 8   TSH                        2870 non-null   float64
 9   T3                         2870 non-null   float64
 10  TT4                        2870 non-null   float64
 11  T4U                        2870 non-null   float64
 12  FTI                        2870 non-null   float64
 13  Thyroid_Class              2870 non-null   int64

In [22]:
# Separate features and target

X = thyroid_model_df.drop(
    "Thyroid_Class",
    axis=1
)

y = thyroid_model_df["Thyroid_Class"]


print("X shape:", X.shape)
print("y shape:", y.shape)


print("\nClass distribution:")
print(y.value_counts())

X shape: (2870, 13)
y shape: (2870,)

Class distribution:
Thyroid_Class
2    2580
1     220
0      70
Name: count, dtype: int64


## Train-Test Split

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


print("\nTraining class distribution:")
print(y_train.value_counts())


print("\nTesting class distribution:")
print(y_test.value_counts())

Training shape: (2296, 13)
Testing shape: (574, 13)

Training class distribution:
Thyroid_Class
2    2064
1     176
0      56
Name: count, dtype: int64

Testing class distribution:
Thyroid_Class
2    516
1     44
0     14
Name: count, dtype: int64


## Training Multiple Classification Models

Multiple machine learning algorithms are trained and compared to identify the best-performing model for thyroid disease classification.

Models evaluated:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting

Performance is compared using accuracy, precision, recall, and F1-score.

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}


model_results = {}

for model_name, model in models.items():

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    model_results[model_name] = {
        "model": model,
        "predictions": y_pred
    }

print("All models trained successfully.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


All models trained successfully.


## Model Evaluation

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd


results = []


for model_name, result in model_results.items():

    y_pred = result["predictions"]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            average="weighted"
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            average="weighted"
        ),
        "F1 Score": f1_score(
            y_test,
            y_pred,
            average="weighted"
        )
    })


results_df = pd.DataFrame(results)

results_df.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
2,Random Forest,0.949477,0.953957,0.949477,0.951680
3,Gradient Boosting,0.949477,0.949507,0.949477,0.949482
1,Decision Tree,0.940767,0.954065,0.940767,0.947075
0,Logistic Regression,0.911150,0.966067,0.911150,0.931684


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

# Select best model
best_model = model_results["Random Forest"]["model"]

# Predictions
rf_predictions = model_results["Random Forest"]["predictions"]


print("Classification Report:\n")

print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=label_encoder.classes_
    )
)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        rf_predictions
    )
)

Classification Report:

              precision    recall  f1-score   support

Hyperthyroid       0.12      0.14      0.13        14
 Hypothyroid       0.98      0.98      0.98        44
      Normal       0.97      0.97      0.97       516

    accuracy                           0.95       574
   macro avg       0.69      0.70      0.69       574
weighted avg       0.95      0.95      0.95       574


Confusion Matrix:
[[  2   0  12]
 [  0  43   1]
 [ 15   1 500]]


# SMOTE

Create synthetic samples for minority classes.

## Handling Class Imbalance Using SMOTE

The thyroid dataset contains significantly fewer Hyperthyroid samples compared to Normal samples.

Synthetic Minority Oversampling Technique (SMOTE) is applied only to the training data to create synthetic samples for minority classes while keeping the test data unchanged for unbiased evaluation.

In [27]:
from imblearn.over_sampling import SMOTE


# Apply SMOTE only on training data

smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


print("Before SMOTE:")
print(y_train.value_counts())


print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

Before SMOTE:
Thyroid_Class
2    2064
1     176
0      56
Name: count, dtype: int64

After SMOTE:
Thyroid_Class
2    2064
1    2064
0    2064
Name: count, dtype: int64


In [28]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score


# Random Forest with SMOTE data

rf_smote = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_smote.fit(
    X_train_smote,
    y_train_smote
)


# Gradient Boosting with SMOTE data

gb_smote = GradientBoostingClassifier(
    random_state=42
)

gb_smote.fit(
    X_train_smote,
    y_train_smote
)


# Predictions

rf_pred_smote = rf_smote.predict(X_test)

gb_pred_smote = gb_smote.predict(X_test)


print("SMOTE models trained successfully")

SMOTE models trained successfully


In [29]:
from sklearn.metrics import classification_report, confusion_matrix


print("Random Forest after SMOTE\n")

print(
    classification_report(
        y_test,
        rf_pred_smote,
        target_names=label_encoder.classes_
    )
)


print("Confusion Matrix - Random Forest")

print(
    confusion_matrix(
        y_test,
        rf_pred_smote
    )
)

Random Forest after SMOTE

              precision    recall  f1-score   support

Hyperthyroid       0.17      0.29      0.22        14
 Hypothyroid       0.96      0.98      0.97        44
      Normal       0.98      0.96      0.97       516

    accuracy                           0.94       574
   macro avg       0.70      0.74      0.72       574
weighted avg       0.96      0.94      0.95       574

Confusion Matrix - Random Forest
[[  4   0  10]
 [  0  43   1]
 [ 19   2 495]]


In [30]:
print("Gradient Boosting after SMOTE\n")

print(
    classification_report(
        y_test,
        gb_pred_smote,
        target_names=label_encoder.classes_
    )
)


print("Confusion Matrix - Gradient Boosting")

print(
    confusion_matrix(
        y_test,
        gb_pred_smote
    )
)

Gradient Boosting after SMOTE

              precision    recall  f1-score   support

Hyperthyroid       0.27      0.57      0.36        14
 Hypothyroid       0.98      0.98      0.98        44
      Normal       0.99      0.96      0.97       516

    accuracy                           0.95       574
   macro avg       0.74      0.83      0.77       574
weighted avg       0.97      0.95      0.96       574

Confusion Matrix - Gradient Boosting
[[  8   0   6]
 [  0  43   1]
 [ 22   1 493]]


## Training XGBoost Model

In [31]:
import xgboost

print(xgboost.__version__)

3.2.0


In [32]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="mlogloss"
)


xgb_model.fit(
    X_train_smote,
    y_train_smote
)


xgb_pred = xgb_model.predict(X_test)


print("XGBoost Classification Report\n")

print(
    classification_report(
        y_test,
        xgb_pred,
        target_names=label_encoder.classes_
    )
)


print("Confusion Matrix")

print(
    confusion_matrix(
        y_test,
        xgb_pred
    )
)

XGBoost Classification Report

              precision    recall  f1-score   support

Hyperthyroid       0.23      0.43      0.30        14
 Hypothyroid       1.00      0.95      0.98        44
      Normal       0.98      0.96      0.97       516

    accuracy                           0.95       574
   macro avg       0.74      0.78      0.75       574
weighted avg       0.96      0.95      0.95       574

Confusion Matrix
[[  6   0   8]
 [  0  42   2]
 [ 20   0 496]]


## Saving Final Thyroid Disease Prediction Model

The Gradient Boosting model trained on SMOTE-balanced data is selected as the final thyroid classifier.

The model, label encoder, and feature list are saved so that the model can be reused later for prediction from blood report inputs.

In [33]:
import joblib


# Save final thyroid model

joblib.dump(
    gb_smote,
    "thyroid_model.pkl"
)


# Save label encoder

joblib.dump(
    label_encoder,
    "thyroid_label_encoder.pkl"
)


# Save feature order

joblib.dump(
    X.columns.tolist(),
    "thyroid_features.pkl"
)


print("Final thyroid model saved successfully.")

Final thyroid model saved successfully.


## Testing Thyroid Model Prediction

In [34]:
import joblib
import pandas as pd


# Load saved files

thyroid_model = joblib.load(
    "thyroid_model.pkl"
)

thyroid_encoder = joblib.load(
    "thyroid_label_encoder.pkl"
)

thyroid_features = joblib.load(
    "thyroid_features.pkl"
)


# Sample patient data

sample_patient = pd.DataFrame(
    [
        {
            "Age": 45,
            "Sex": 0,
            "On_Thyroxine": 0,
            "On_Antithyroid_Medication": 0,
            "Query_Hypothyroid": 1,
            "Query_Hyperthyroid": 0,
            "Thyroid_Surgery": 0,
            "I131_Treatment": 0,
            "TSH": 8.5,
            "T3": 0.8,
            "TT4": 70,
            "T4U": 0.9,
            "FTI": 78
        }
    ]
)


# Arrange columns in correct order

sample_patient = sample_patient[thyroid_features]


# Prediction

prediction = thyroid_model.predict(sample_patient)


# Convert number back to class name

result = thyroid_encoder.inverse_transform(
    prediction
)


print("Predicted Thyroid Condition:", result[0])

Predicted Thyroid Condition: Hypothyroid


In [35]:
import os

os.listdir("/kaggle/working")

['thyroid_model.pkl',
 'thyroid_label_encoder.pkl',
 '.virtual_documents',
 'thyroid_features.pkl']

In [36]:
import os

for root, dirs, files in os.walk("/kaggle"):
    for file in files:
        if file.endswith(".pkl"):
            print(os.path.join(root, file))

/kaggle/working/thyroid_model.pkl
/kaggle/working/thyroid_label_encoder.pkl
/kaggle/working/thyroid_features.pkl
